# BioNNE-L: RU Dictionary Pretraining for the Cross-Encoder

This notebook pretrains a cross-encoder on UMLS dictionary data for the Russian track. The resulting checkpoint can be used as `RERANKER_MODEL_NAME_OR_PATH` in `bionnel-cross-encoder-reranking.ipynb` before supervised entity-linking fine-tuning.


In [ ]:
import copy
import gc
import json
import logging
import os
import random
import shutil
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from lib.data.text_preprocessing import preprocess_text
from lib.data.vocab_enrichment import enrich_vocab_with_oov_train_dev_terms, prepare_experiment_vocab
from lib.retrieval.reranking.candidate_context import build_candidate_context_cache
from lib.retrieval.reranking.candidate_context_cache import (
    build_candidate_text_map,
    load_candidate_context_cache,
    save_candidate_context_cache,
)
from lib.retrieval.reranking.dictionary_pretrain.artifacts import (
    build_dictionary_pretrain_cache_metadata,
    load_dataframe_cache,
    save_dataframe_cache,
)
from lib.retrieval.reranking.dictionary_pretrain.data import (
    assign_dictionary_pretrain_splits,
    build_dictionary_pretrain_concepts,
    build_dictionary_pretrain_queries,
    build_dictionary_pretrain_vocab_subset,
)
from lib.retrieval.reranking.dictionary_pretrain.fingerprints import (
    fingerprint_candidate_text_map,
    fingerprint_dictionary_pretrain_dataframe,
)
from lib.retrieval.reranking.dictionary_pretrain.model_io import load_cross_encoder_from_pretrained
from lib.retrieval.reranking.dictionary_pretrain.pairwise import build_dictionary_pretrain_pairwise_data
from lib.retrieval.reranking.dictionary_pretrain.retriever_cache import (
    build_dictionary_pretrain_retriever_cache,
    infer_dictionary_pretrain_cache_topk,
    load_dictionary_pretrain_retriever_cache,
    save_dictionary_pretrain_retriever_cache,
    trim_dictionary_pretrain_retriever_cache,
)
from lib.retrieval.reranking.dictionary_pretrain.training import train_dictionary_pretrain_cross_encoder
from lib.retrieval.reranking.io import save_json
from lib.utils.logging_utils import configure_logging


In [ ]:
configure_logging(level=logging.INFO, force=True)

ARTIFACTS_DIR = './artifacts_cross_encoder_dictionary_pretrain'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

MLFLOW_EXPERIMENT_PREFIX = 'bionnel-cross-encoder-dict-pretrain-ru'
MLFLOW_RUN_NAME_TEMPLATE = '{dataset_name}-bce'

DEFAULT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEFAULT_DEVICE


In [ ]:
LOCAL_MLRUNS_DIR = Path('mlruns').resolve()
mlflow.set_tracking_uri(LOCAL_MLRUNS_DIR.as_uri())
print('MLflow tracking URI:', mlflow.get_tracking_uri())


In [ ]:
## Common Hyperparameters
COMMON_TRAINING_CONFIG = {
    'SEED': 42,  # Random seed for splitting and training.
    'EPOCHS': 5,  # Number of dictionary-pretraining epochs.
    'TRAIN_BATCH_SIZE': 128,  # Per-device training batch size.
    'EVAL_BATCH_SIZE': 128,  # Per-device evaluation batch size.
    'LEARNING_RATE': 2e-5,  # Optimizer learning rate.
    'WEIGHT_DECAY': 0.01,  # Optimizer weight decay.
    'WARMUP_RATIO': 0.1,  # Warmup fraction for the learning-rate schedule.
    'MAX_SEQ_LENGTH': 384,  # Maximum token length for cross-encoder pairs.
    'GRAD_ACCUMULATION_STEPS': 1,  # Gradient accumulation steps.
    'TRAIN_LOGGING_STEPS': 25,  # Training log interval in optimizer steps.
    'LOSS_NAME': 'bce',  # Dictionary pretraining loss.
    'SELECTION_METRIC': 'Acc@1',  # Dev metric used to select the best checkpoint.
    'RERANK_BATCH_SIZE': 64,  # CrossEncoder.predict batch size for dev ranking evaluation.
}

COMMON_RETRIEVAL_CONFIG = {
    'ENRICH_VOCABULARY': False,  # Optionally add all unique train/dev mention-CUI pairs beyond the default OOV-only step.
    'DEDUPLICATE_BY_CUI': True,  # Keep at most one candidate per CUI in retriever top-k results.
    'PRETRAIN_TOPK': 25,  # Retriever candidates scored per pseudo-query.
    'PRETRAIN_NUM_NEGATIVES': 20,  # Negative candidates kept per pseudo-query for pairwise data.
    'QUERY_BATCH_SIZE': 262_144,  # Number of pseudo-queries processed per dense retrieval batch.
    'DENSE_VOCAB_BATCH_SIZE': 16_384,  # Vocabulary chunk size for dense scoring.
    'ST_ENCODE_BATCH_SIZE': 1024,  # SentenceTransformer encoding batch size for retrieval.
}

COMMON_CANDIDATE_CONTEXT_CONFIG = {
    'ENABLED': True,  # Use candidate-context profiles instead of raw candidate names only.
    'ALIAS_LENGTH_THRESHOLD': None,  # If None, auto-estimate alias length cap from vocabulary statistics.
    'MAX_ALIASES': 6,  # Maximum number of non-representative aliases in a candidate profile.
    'GROUP_LIMITS': {
        'abbreviations': 1,  # Maximum abbreviation aliases per candidate profile.
        'short_names': 2,  # Maximum short-name aliases per candidate profile.
        'multi_word': 2,  # Maximum multi-word aliases per candidate profile.
        'long_variants': 1,  # Maximum long-variant aliases per candidate profile.
    },
    'PREFERRED_LANGUAGES': ['RUS', 'ENG'],  # Preferred alias languages for candidate profiles.
    'ALLOWED_LANGUAGES': [],  # Empty keeps all languages after preference ordering.
    'NUM_WORKERS': 4,  # Parallel workers for candidate-context construction.
    'LOAD_FROM_DISK_IF_AVAILABLE': True,  # Reuse candidate-context cache when metadata matches.
    'FORCE_REBUILD': False,  # Ignore candidate-context cache and rebuild from vocabulary.
    'CACHE_STEM': 'candidate_context_short',  # Artifact stem for candidate-context cache files.
}

COMMON_PRETRAIN_CONFIG = {
    'MAX_PSEUDO_QUERIES_PER_CUI': 5,  # Maximum synonym pseudo-queries sampled per CUI.
    'MIN_PSEUDO_QUERIES_PER_CUI': 1,  # Minimum pseudo-queries required to keep a CUI.
    'PREFERRED_QUERY_LANGUAGES': ['RUS', 'ENG'],  # Preferred synonym languages for pseudo-queries.
    'NUM_EXTRA_RANDOM_CUIS': 25000,  # Extra random CUIs added beyond train/dev and Russian CUIs.
    'SIZE_ESTIMATE_NUM_NEGATIVES': 20,  # Negatives per query used by the optional size estimator.
    'PAIRWISE_NUM_WORKERS': 4,  # Parallel workers for pairwise example construction.
    'VALIDATION_FRACTION': 0.05,  # Fraction of concepts assigned to the dictionary-pretrain dev split.
    'LOAD_FROM_DISK_IF_AVAILABLE': True,  # Reuse intermediate pretraining caches when metadata matches.
    'FORCE_REBUILD_CONCEPTS': False,  # Rebuild concept-level pseudo-query cache.
    'FORCE_REBUILD_QUERIES': False,  # Rebuild query-level pseudo-query cache.
    'FORCE_REBUILD_RETRIEVER_CACHE': False,  # Rebuild dense retriever cache for pseudo-queries.
    'FORCE_REBUILD_PAIRWISE': False,  # Rebuild pairwise cross-encoder examples.
    'CONCEPT_CACHE_STEM': 'pretrain_concepts',  # Artifact stem for concept-level cache.
    'QUERY_CACHE_STEM': 'pretrain_queries',  # Artifact stem for query-level cache.
    'RETRIEVER_CACHE_STEM': 'pretrain_retriever_cache',  # Artifact stem for retriever cache.
    'PAIRWISE_CACHE_STEM': 'pretrain_pair_examples',  # Artifact stem for pairwise examples.
}


## Data Loading

In [ ]:
ru_data_train = pd.read_parquet('data/parquet/ru/bionnel_ru_train.parquet')
ru_data_dev = pd.read_parquet('data/parquet/ru/bionnel_ru_dev.parquet')

raw_vocab = pd.read_parquet('data/vocabular/bionnel_vocab_bilingual.parquet')
normalized_vocab = raw_vocab.copy()

for dataset_df in [ru_data_train, ru_data_dev]:
    dataset_df['raw_text'] = dataset_df['text'].astype(str)
    dataset_df['text'] = dataset_df['text'].map(preprocess_text)

normalized_vocab['concept_name'] = normalized_vocab['concept_name'].map(preprocess_text)

RU_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat([
    ru_data_train,
    ru_data_dev,
], ignore_index=True)

RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT = RU_ENTITIES_FOR_VOCAB_ENRICHMENT

print('RU train/dev:', ru_data_train.shape, ru_data_dev.shape)
print('Base vocabulary shape:', normalized_vocab.shape)
print('RU vocab enrichment pool:', RU_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print('RU raw candidate-context enrichment pool:', RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT.shape)
print('Semantic types:', sorted(normalized_vocab['semantic_type'].dropna().unique().tolist()))


## Training Utils

In [ ]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_artifact_dir(dataset_name):
    artifact_dir = Path(ARTIFACTS_DIR) / dataset_name.lower()
    artifact_dir.mkdir(parents=True, exist_ok=True)
    return artifact_dir


def resolve_default_retriever_model_path(dataset_name):
    return get_artifact_dir(dataset_name).parent.parent / 'artifacts_dense_finetuning' / dataset_name.lower() / 'training' / 'best_model'


def build_base_experiment_config(reranker_model_name_or_path, dataset_name, retriever_model_path=None):
    retriever_model_path = retriever_model_path or resolve_default_retriever_model_path(dataset_name)
    return {
        'RERANKER_MODEL_NAME_OR_PATH': reranker_model_name_or_path,
        'DEVICE': DEFAULT_DEVICE,
        'TRAINING': copy.deepcopy(COMMON_TRAINING_CONFIG),
        'RETRIEVAL': copy.deepcopy(COMMON_RETRIEVAL_CONFIG),
        'CANDIDATE_CONTEXT': copy.deepcopy(COMMON_CANDIDATE_CONTEXT_CONFIG),
        'PRETRAIN': copy.deepcopy(COMMON_PRETRAIN_CONFIG),
        'RETRIEVER_MODEL_NAME_OR_PATH': str(retriever_model_path),
    }


def build_runtime_config(cfg):
    runtime_cfg = {
        'RERANKER_MODEL_NAME_OR_PATH': cfg['RERANKER_MODEL_NAME_OR_PATH'],
        'DEVICE': cfg['DEVICE'],
        'RETRIEVER_MODEL_NAME_OR_PATH': cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
    }
    runtime_cfg.update(copy.deepcopy(cfg['TRAINING']))
    runtime_cfg.update(copy.deepcopy(cfg['RETRIEVAL']))
    runtime_cfg['TRAINING'] = copy.deepcopy(cfg['TRAINING'])
    runtime_cfg['RETRIEVAL'] = copy.deepcopy(cfg['RETRIEVAL'])
    runtime_cfg['CANDIDATE_CONTEXT'] = copy.deepcopy(cfg['CANDIDATE_CONTEXT'])
    runtime_cfg['PRETRAIN'] = copy.deepcopy(cfg['PRETRAIN'])
    return runtime_cfg


def build_effective_experiment_config(base_cfg, training_result, candidate_artifact_paths, candidate_metadata, pretrain_cache_artifacts, pretrain_cache_metadata, pretrain_metrics):
    effective_cfg = copy.deepcopy(base_cfg)
    effective_cfg['TRAINING'].update({
        'LOSS_NAME': str(base_cfg['TRAINING'].get('LOSS_NAME', 'bce')),
        'SELECTION_METRIC': str(base_cfg['TRAINING'].get('SELECTION_METRIC', 'Acc@1')),
        'BEST_EPOCH': None if training_result.get('best_epoch') is None else int(training_result['best_epoch']),
        'BEST_MODEL_DIR': str(training_result['best_model_dir']),
        'BEST_CHECKPOINT_DIR': str(training_result['best_checkpoint_dir']),
        'USED_EVAL_SPLIT': bool(training_result['used_eval_split']),
        'NUM_TRAIN_PAIRS': int(len(training_result['train_pair_examples_df'])),
        'NUM_EVAL_PAIRS': int(len(training_result['eval_pair_examples_df'])),
    })
    effective_cfg['CANDIDATE_CONTEXT']['ARTIFACTS'] = candidate_artifact_paths
    effective_cfg['CANDIDATE_CONTEXT']['METADATA'] = candidate_metadata
    effective_cfg['PRETRAIN']['CACHE_ARTIFACTS'] = pretrain_cache_artifacts
    effective_cfg['PRETRAIN']['CACHE_METADATA'] = pretrain_cache_metadata
    effective_cfg['PRETRAIN']['METRICS'] = pretrain_metrics
    return effective_cfg


def load_retriever_model(model_name_or_path, device=None):
    model_path = Path(model_name_or_path)
    resolved_model = str(model_path) if model_path.exists() else str(model_name_or_path)
    resolved_device = DEFAULT_DEVICE if device is None else device
    return SentenceTransformer(resolved_model, device=resolved_device)


def prepare_candidate_context_vocab(raw_base_vocab_df, enrichment_entities_df, cfg):
    return prepare_experiment_vocab(
        raw_base_vocab_df,
        enrichment_entities_df,
        cfg['RETRIEVAL'],
        text_column='raw_text',
    )


def build_random_pairwise_size_estimate(vocab_df, cfg):
    pretrain_cfg = cfg['PRETRAIN']
    training_cfg = cfg['TRAINING']

    concepts_df = build_dictionary_pretrain_concepts(
        vocab_df=vocab_df,
        candidate_text_map=None,
        max_pseudo_queries_per_cui=pretrain_cfg['MAX_PSEUDO_QUERIES_PER_CUI'],
        min_pseudo_queries_per_cui=pretrain_cfg['MIN_PSEUDO_QUERIES_PER_CUI'],
        preferred_query_languages=pretrain_cfg.get('PREFERRED_QUERY_LANGUAGES', []),
    )
    split_concepts_df = assign_dictionary_pretrain_splits(
        concepts_df,
        validation_fraction=pretrain_cfg['VALIDATION_FRACTION'],
        seed=training_cfg['SEED'],
    )
    queries_df = build_dictionary_pretrain_queries(split_concepts_df)

    negatives_per_type = (
        split_concepts_df.groupby('semantic_type', sort=False)['CUI']
        .nunique()
        .sub(1)
        .clip(lower=0)
        .rename('available_negatives')
        .reset_index()
    )
    query_counts_by_type = (
        queries_df.groupby('semantic_type', sort=False)['query_id']
        .size()
        .rename('num_queries')
        .reset_index()
    )
    estimate_df = query_counts_by_type.merge(negatives_per_type, on='semantic_type', how='left')
    estimate_df['available_negatives'] = estimate_df['available_negatives'].fillna(0).astype(int)
    estimate_df['negatives_per_query'] = estimate_df['available_negatives'].clip(
        upper=int(pretrain_cfg['SIZE_ESTIMATE_NUM_NEGATIVES'])
    )
    estimate_df['num_positive_pairs'] = estimate_df['num_queries'].astype(int)
    estimate_df['num_negative_pairs'] = (
        estimate_df['num_queries'].astype(int) * estimate_df['negatives_per_query'].astype(int)
    )
    estimate_df['num_pairs'] = estimate_df['num_positive_pairs'] + estimate_df['num_negative_pairs']

    total_queries = int(len(queries_df))
    total_positive_pairs = int(estimate_df['num_positive_pairs'].sum())
    total_negative_pairs = int(estimate_df['num_negative_pairs'].sum())
    total_pairs = int(estimate_df['num_pairs'].sum())

    avg_query_chars = float(queries_df['query_text'].astype(str).str.len().mean()) if not queries_df.empty else 0.0
    avg_candidate_chars = float(split_concepts_df['candidate_text'].astype(str).str.len().mean()) if not split_concepts_df.empty else 0.0
    bytes_per_char = 2.0
    replicated_text_bytes = total_pairs * (avg_query_chars + avg_candidate_chars) * bytes_per_char
    overhead_multiplier = 2.5
    estimated_pairwise_ram_gb = (replicated_text_bytes * overhead_multiplier) / (1024 ** 3)

    stats = {
        'num_concepts': int(len(split_concepts_df)),
        'num_queries': total_queries,
        'num_pairs': total_pairs,
        'num_positive_pairs': total_positive_pairs,
        'num_negative_pairs': total_negative_pairs,
        'avg_queries_per_concept': float(total_queries / max(len(split_concepts_df), 1)),
        'avg_pairs_per_query': float(total_pairs / max(total_queries, 1)),
        'avg_query_chars': avg_query_chars,
        'avg_candidate_chars': avg_candidate_chars,
        'estimated_pairwise_ram_gb': float(estimated_pairwise_ram_gb),
    }
    return {
        'concepts_df': split_concepts_df,
        'queries_df': queries_df,
        'estimate_df': estimate_df.sort_values('num_pairs', ascending=False, kind='stable').reset_index(drop=True),
        'stats': stats,
    }


def _load_cache_metadata(metadata_path):
    metadata_path = Path(metadata_path)
    if not metadata_path.exists():
        return None
    return json.loads(metadata_path.read_text(encoding='utf-8'))


def _can_reuse_cache(metadata_path, expected_metadata):
    existing_metadata = _load_cache_metadata(metadata_path)
    return existing_metadata == expected_metadata


def _can_reuse_retriever_cache(metadata_path, expected_metadata, requested_topk):
    existing_metadata = _load_cache_metadata(metadata_path)
    if existing_metadata is None:
        return False, None

    existing_cfg = dict(existing_metadata.get('config', {}))
    expected_cfg = dict(expected_metadata.get('config', {}))
    existing_topk = int(existing_cfg.pop('pretrain_topk', 0) or 0)
    expected_topk = int(expected_cfg.pop('pretrain_topk', 0) or 0)

    if existing_metadata.get('vocab_fingerprint') != expected_metadata.get('vocab_fingerprint'):
        return False, existing_metadata
    if existing_cfg != expected_cfg:
        return False, existing_metadata
    if existing_topk < int(requested_topk):
        return False, existing_metadata
    if existing_topk < expected_topk:
        return False, existing_metadata
    return True, existing_metadata


def prepare_candidate_context_cache_for_experiment(vocab_df, dataset_name, cfg):
    if not cfg['CANDIDATE_CONTEXT'].get('ENABLED', False):
        return None, {}, {}, {}

    artifact_dir = get_artifact_dir(dataset_name)
    cache_stem = cfg['CANDIDATE_CONTEXT'].get('CACHE_STEM', 'candidate_context')
    cache_path = artifact_dir / f'{cache_stem}.parquet'
    metadata_path = artifact_dir / f'{cache_stem}_metadata.json'

    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'alias_length_threshold': cfg['CANDIDATE_CONTEXT'].get('ALIAS_LENGTH_THRESHOLD'),
            'max_aliases': cfg['CANDIDATE_CONTEXT'].get('MAX_ALIASES', 8),
            'group_limits': cfg['CANDIDATE_CONTEXT'].get('GROUP_LIMITS'),
            'preferred_languages': cfg['CANDIDATE_CONTEXT'].get('PREFERRED_LANGUAGES'),
            'allowed_languages': cfg['CANDIDATE_CONTEXT'].get('ALLOWED_LANGUAGES'),
            'cache_stem': cache_stem,
        },
    )

    use_disk_cache = (
        cfg['CANDIDATE_CONTEXT'].get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not cfg['CANDIDATE_CONTEXT'].get('FORCE_REBUILD', False)
        and cache_path.exists()
        and _can_reuse_cache(metadata_path, expected_metadata)
    )

    if use_disk_cache:
        candidate_context_df, _ = load_candidate_context_cache(cache_path, metadata_path)
        artifact_paths = {
            'cache_path': str(cache_path),
            'preview_path': str(artifact_dir / f'{cache_stem}_preview.parquet'),
            'metadata_path': str(metadata_path),
        }
    else:
        candidate_context_df, _ = build_candidate_context_cache(
            vocab_df=vocab_df,
            alias_length_threshold=cfg['CANDIDATE_CONTEXT'].get('ALIAS_LENGTH_THRESHOLD'),
            max_aliases=cfg['CANDIDATE_CONTEXT'].get('MAX_ALIASES', 8),
            group_limits=cfg['CANDIDATE_CONTEXT'].get('GROUP_LIMITS'),
            preferred_languages=cfg['CANDIDATE_CONTEXT'].get('PREFERRED_LANGUAGES'),
            allowed_languages=cfg['CANDIDATE_CONTEXT'].get('ALLOWED_LANGUAGES'),
            num_workers=cfg['CANDIDATE_CONTEXT'].get('NUM_WORKERS', 1),
        )
        artifact_paths = save_candidate_context_cache(candidate_context_df, expected_metadata, artifact_dir, stem=cache_stem)

    candidate_metadata = dict(expected_metadata)
    if candidate_context_df is not None and not candidate_context_df.empty and 'alias_length_threshold' in candidate_context_df.columns:
        candidate_metadata['threshold'] = int(candidate_context_df['alias_length_threshold'].iloc[0])
    candidate_metadata['num_candidate_rows'] = 0 if candidate_context_df is None else int(len(candidate_context_df))
    candidate_metadata['loaded_from_disk'] = bool(use_disk_cache)
    candidate_text_map = build_candidate_text_map(candidate_context_df)
    return candidate_context_df, candidate_metadata, artifact_paths, candidate_text_map


def prepare_pretrain_concepts_cache(dataset_name, vocab_df, cfg, candidate_text_map):
    artifact_dir = get_artifact_dir(dataset_name)
    pretrain_cfg = cfg['PRETRAIN']
    stem = pretrain_cfg['CONCEPT_CACHE_STEM']
    cache_path = artifact_dir / f'{stem}.parquet'
    metadata_path = artifact_dir / f'{stem}_metadata.json'
    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'max_pseudo_queries_per_cui': pretrain_cfg['MAX_PSEUDO_QUERIES_PER_CUI'],
            'min_pseudo_queries_per_cui': pretrain_cfg['MIN_PSEUDO_QUERIES_PER_CUI'],
            'candidate_text_map_fingerprint': fingerprint_candidate_text_map(candidate_text_map),
            'preferred_query_languages': pretrain_cfg.get('PREFERRED_QUERY_LANGUAGES', []),
        },
    )
    use_disk_cache = (
        pretrain_cfg.get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not pretrain_cfg.get('FORCE_REBUILD_CONCEPTS', False)
        and cache_path.exists()
        and _can_reuse_cache(metadata_path, expected_metadata)
    )

    if use_disk_cache:
        concepts_df, _ = load_dataframe_cache(cache_path, metadata_path)
    else:
        concepts_df = build_dictionary_pretrain_concepts(
            vocab_df=vocab_df,
            candidate_text_map=candidate_text_map,
            max_pseudo_queries_per_cui=pretrain_cfg['MAX_PSEUDO_QUERIES_PER_CUI'],
            min_pseudo_queries_per_cui=pretrain_cfg['MIN_PSEUDO_QUERIES_PER_CUI'],
            preferred_query_languages=pretrain_cfg.get('PREFERRED_QUERY_LANGUAGES', []),
        )
        save_dataframe_cache(concepts_df, expected_metadata, artifact_dir, stem=stem)

    metadata = dict(expected_metadata)
    metadata['loaded_from_disk'] = bool(use_disk_cache)
    artifact_paths = {
        'cache_path': str(cache_path),
        'preview_path': str(artifact_dir / f'{stem}_preview.tsv'),
        'metadata_path': str(metadata_path),
    }
    return concepts_df, metadata, artifact_paths


def prepare_pretrain_queries_cache(dataset_name, vocab_df, concepts_df, cfg):
    artifact_dir = get_artifact_dir(dataset_name)
    pretrain_cfg = cfg['PRETRAIN']
    stem = pretrain_cfg['QUERY_CACHE_STEM']
    cache_path = artifact_dir / f'{stem}.parquet'
    metadata_path = artifact_dir / f'{stem}_metadata.json'
    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'validation_fraction': pretrain_cfg['VALIDATION_FRACTION'],
            'seed': cfg['TRAINING']['SEED'],
            'concepts_fingerprint': fingerprint_dictionary_pretrain_dataframe(concepts_df, columns=['CUI', 'semantic_type', 'candidate_text', 'pseudo_queries']),
        },
    )
    use_disk_cache = (
        pretrain_cfg.get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not pretrain_cfg.get('FORCE_REBUILD_QUERIES', False)
        and cache_path.exists()
        and _can_reuse_cache(metadata_path, expected_metadata)
    )

    if use_disk_cache:
        queries_df, _ = load_dataframe_cache(cache_path, metadata_path)
    else:
        split_concepts_df = assign_dictionary_pretrain_splits(
            concepts_df,
            validation_fraction=pretrain_cfg['VALIDATION_FRACTION'],
            seed=cfg['TRAINING']['SEED'],
        )
        queries_df = build_dictionary_pretrain_queries(split_concepts_df)
        save_dataframe_cache(queries_df, expected_metadata, artifact_dir, stem=stem)

    metadata = dict(expected_metadata)
    metadata['loaded_from_disk'] = bool(use_disk_cache)
    artifact_paths = {
        'cache_path': str(cache_path),
        'preview_path': str(artifact_dir / f'{stem}_preview.tsv'),
        'metadata_path': str(metadata_path),
    }
    return queries_df, metadata, artifact_paths


def prepare_pretrain_retriever_cache(dataset_name, vocab_df, queries_df, cfg):
    artifact_dir = get_artifact_dir(dataset_name)
    pretrain_cfg = cfg['PRETRAIN']
    retrieval_cfg = cfg['RETRIEVAL']
    stem = pretrain_cfg['RETRIEVER_CACHE_STEM']
    cache_path = artifact_dir / f'{stem}.pkl'
    metadata_path = artifact_dir / f'{stem}_metadata.json'
    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'retriever_model_name_or_path': cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
            'pretrain_topk': retrieval_cfg['PRETRAIN_TOPK'],
            'deduplicate_by_cui': retrieval_cfg['DEDUPLICATE_BY_CUI'],
            'queries_fingerprint': fingerprint_dictionary_pretrain_dataframe(queries_df, columns=['query_id', 'query_text', 'CUI', 'semantic_type', 'split']),
        },
    )
    use_disk_cache = False
    existing_metadata = None
    if (
        pretrain_cfg.get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not pretrain_cfg.get('FORCE_REBUILD_RETRIEVER_CACHE', False)
        and cache_path.exists()
    ):
        use_disk_cache, existing_metadata = _can_reuse_retriever_cache(
            metadata_path,
            expected_metadata,
            retrieval_cfg['PRETRAIN_TOPK'],
        )

    if use_disk_cache:
        retriever_cache = load_dictionary_pretrain_retriever_cache(cache_path)
        cached_topk = infer_dictionary_pretrain_cache_topk(retriever_cache)
        if cached_topk > int(retrieval_cfg['PRETRAIN_TOPK']):
            retriever_cache = trim_dictionary_pretrain_retriever_cache(
                retriever_cache,
                topk=retrieval_cfg['PRETRAIN_TOPK'],
            )
    else:
        retriever_model = load_retriever_model(cfg['RETRIEVER_MODEL_NAME_OR_PATH'], device=cfg['DEVICE'])
        retriever_cache = build_dictionary_pretrain_retriever_cache(
            queries_df=queries_df,
            vocab_df=vocab_df,
            retriever_model=retriever_model,
            topk=retrieval_cfg['PRETRAIN_TOPK'],
            query_batch_size=retrieval_cfg['QUERY_BATCH_SIZE'],
            dense_vocab_batch_size=retrieval_cfg['DENSE_VOCAB_BATCH_SIZE'],
            st_encode_batch_size=retrieval_cfg['ST_ENCODE_BATCH_SIZE'],
            deduplicate_by_cui=retrieval_cfg['DEDUPLICATE_BY_CUI'],
        )
        save_dictionary_pretrain_retriever_cache(retriever_cache, artifact_dir, stem=stem)
        save_json(expected_metadata, metadata_path)
        del retriever_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metadata = dict(expected_metadata)
    metadata['loaded_from_disk'] = bool(use_disk_cache)
    metadata['reused_from_cached_topk'] = None if not use_disk_cache else int((existing_metadata or {}).get('config', {}).get('pretrain_topk', retrieval_cfg['PRETRAIN_TOPK']))
    artifact_paths = {
        'pickle_path': str(cache_path),
        'preview_path': str(artifact_dir / f'{stem}.tsv'),
        'metadata_path': str(metadata_path),
    }
    return retriever_cache, metadata, artifact_paths


def prepare_pretrain_pairwise_cache(dataset_name, vocab_df, queries_df, retriever_cache, candidate_text_map, cfg):
    artifact_dir = get_artifact_dir(dataset_name)
    pretrain_cfg = cfg['PRETRAIN']
    retrieval_cfg = cfg['RETRIEVAL']
    stem = pretrain_cfg['PAIRWISE_CACHE_STEM']
    cache_path = artifact_dir / f'{stem}.parquet'
    metadata_path = artifact_dir / f'{stem}_metadata.json'
    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'num_negatives': retrieval_cfg['PRETRAIN_NUM_NEGATIVES'],
            'queries_fingerprint': fingerprint_dictionary_pretrain_dataframe(queries_df, columns=['query_id', 'query_text', 'CUI', 'semantic_type', 'candidate_text', 'split']),
            'candidate_text_map_fingerprint': fingerprint_candidate_text_map(candidate_text_map),
            'preferred_query_languages': pretrain_cfg.get('PREFERRED_QUERY_LANGUAGES', []),
        },
    )
    use_disk_cache = (
        pretrain_cfg.get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not pretrain_cfg.get('FORCE_REBUILD_PAIRWISE', False)
        and cache_path.exists()
        and _can_reuse_cache(metadata_path, expected_metadata)
    )

    if use_disk_cache:
        pairwise_df, _ = load_dataframe_cache(cache_path, metadata_path)
    else:
        pairwise_df = build_dictionary_pretrain_pairwise_data(
            queries_df=queries_df,
            retriever_cache=retriever_cache,
            candidate_text_map=candidate_text_map,
            num_negatives=retrieval_cfg['PRETRAIN_NUM_NEGATIVES'],
            num_workers=pretrain_cfg.get('PAIRWISE_NUM_WORKERS', 1),
        )
        save_dataframe_cache(pairwise_df, expected_metadata, artifact_dir, stem=stem)

    metadata = dict(expected_metadata)
    metadata['loaded_from_disk'] = bool(use_disk_cache)
    artifact_paths = {
        'cache_path': str(cache_path),
        'preview_path': str(artifact_dir / f'{stem}_preview.tsv'),
        'metadata_path': str(metadata_path),
    }
    return pairwise_df, metadata, artifact_paths


def run_pretraining(dataset_name, query_vocab_df, retrieval_vocab_df, candidate_context_vocab_df, cfg):
    runtime_cfg = build_runtime_config(cfg)
    set_seed(runtime_cfg['SEED'])

    candidate_context_df, candidate_context_metadata, candidate_context_artifact_paths, candidate_text_map = prepare_candidate_context_cache_for_experiment(
        vocab_df=candidate_context_vocab_df,
        dataset_name=dataset_name,
        cfg=runtime_cfg,
    )

    concepts_df, concepts_metadata, concepts_artifact_paths = prepare_pretrain_concepts_cache(
        dataset_name=dataset_name,
        vocab_df=query_vocab_df,
        cfg=runtime_cfg,
        candidate_text_map=candidate_text_map,
    )

    queries_df, queries_metadata, queries_artifact_paths = prepare_pretrain_queries_cache(
        dataset_name=dataset_name,
        vocab_df=query_vocab_df,
        concepts_df=concepts_df,
        cfg=runtime_cfg,
    )

    retriever_cache, retriever_metadata, retriever_artifact_paths = prepare_pretrain_retriever_cache(
        dataset_name=dataset_name,
        vocab_df=retrieval_vocab_df,
        queries_df=queries_df,
        cfg=runtime_cfg,
    )

    pairwise_df, pairwise_metadata, pairwise_artifact_paths = prepare_pretrain_pairwise_cache(
        dataset_name=dataset_name,
        vocab_df=retrieval_vocab_df,
        queries_df=queries_df,
        retriever_cache=retriever_cache,
        candidate_text_map=candidate_text_map,
        cfg=runtime_cfg,
    )

    training_result = train_dictionary_pretrain_cross_encoder(
        pairwise_df=pairwise_df,
        cross_encoder_model_name=runtime_cfg['RERANKER_MODEL_NAME_OR_PATH'],
        output_dir=get_artifact_dir(dataset_name) / 'training',
        cfg=runtime_cfg['TRAINING'] | {'DEVICE': runtime_cfg['DEVICE']},
    )

    cache_artifacts = {
        'candidate_context': candidate_context_artifact_paths,
        'pretrain_concepts': concepts_artifact_paths,
        'pretrain_queries': queries_artifact_paths,
        'pretrain_retriever_cache': retriever_artifact_paths,
        'pretrain_pair_examples': pairwise_artifact_paths,
    }
    cache_metadata = {
        'candidate_context': candidate_context_metadata,
        'pretrain_concepts': concepts_metadata,
        'pretrain_queries': queries_metadata,
        'pretrain_retriever_cache': retriever_metadata,
        'pretrain_pair_examples': pairwise_metadata,
    }

    return {
        'runtime_cfg': runtime_cfg,
        'candidate_context_df': candidate_context_df,
        'candidate_context_metadata': candidate_context_metadata,
        'candidate_text_map': candidate_text_map,
        'concepts_df': concepts_df,
        'queries_df': queries_df,
        'retriever_cache': retriever_cache,
        'pairwise_df': pairwise_df,
        'training_result': training_result,
        'cache_artifacts': cache_artifacts,
        'cache_metadata': cache_metadata,
    }


## Validation Utils

In [ ]:
def load_best_model(effective_cfg):
    return load_cross_encoder_from_pretrained(
        effective_cfg['TRAINING']['BEST_MODEL_DIR'],
        device=effective_cfg['DEVICE'],
    )


def summarize_pairwise_split(pairwise_df, split_name):
    split_df = pairwise_df[pairwise_df['split'].astype(str) == str(split_name)].copy()
    if split_df.empty:
        return {
            'split': split_name,
            'num_queries': 0,
            'num_pairs': 0,
            'num_positive_pairs': 0,
            'num_negative_pairs': 0,
        }
    return {
        'split': split_name,
        'num_queries': int(split_df['query_id'].nunique()),
        'num_pairs': int(len(split_df)),
        'num_positive_pairs': int((split_df['label'] > 0).sum()),
        'num_negative_pairs': int((split_df['label'] <= 0).sum()),
    }


def compute_retriever_hit_rate(pairwise_df, split_name):
    split_df = pairwise_df[pairwise_df['split'].astype(str) == str(split_name)].copy()
    if split_df.empty:
        return None
    query_has_negative = split_df.groupby('query_id')['candidate_rank'].apply(lambda values: any(int(value) > 0 for value in values)).astype(bool)
    return float(query_has_negative.mean()) if len(query_has_negative) else None





## MLflow Utils

In [ ]:
def flatten_config_for_mlflow(prefix, value):
    if isinstance(value, dict):
        flat = {}
        for key, nested_value in value.items():
            nested_prefix = f'{prefix}.{key}' if prefix else str(key)
            flat.update(flatten_config_for_mlflow(nested_prefix, nested_value))
        return flat
    if isinstance(value, (list, tuple)):
        return {prefix: str(list(value))}
    return {prefix: value}


def build_mlflow_params(dataset_name, base_cfg, effective_cfg):
    training_cfg = effective_cfg['TRAINING']
    retrieval_cfg = effective_cfg['RETRIEVAL']
    candidate_context_cfg = effective_cfg['CANDIDATE_CONTEXT']
    pretrain_cfg = effective_cfg['PRETRAIN']
    return {
        'dataset_name': dataset_name,
        'reranker_model_name_or_path': effective_cfg['RERANKER_MODEL_NAME_OR_PATH'],
        'retriever_model_name_or_path': effective_cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
        'device': effective_cfg['DEVICE'],
        'seed': int(training_cfg['SEED']),
        'epochs': int(training_cfg['EPOCHS']),
        'train_batch_size': int(training_cfg['TRAIN_BATCH_SIZE']),
        'eval_batch_size': int(training_cfg['EVAL_BATCH_SIZE']),
        'learning_rate': float(training_cfg['LEARNING_RATE']),
        'weight_decay': float(training_cfg['WEIGHT_DECAY']),
        'warmup_ratio': float(training_cfg['WARMUP_RATIO']),
        'max_seq_length': int(training_cfg['MAX_SEQ_LENGTH']),
        'grad_accumulation_steps': int(training_cfg['GRAD_ACCUMULATION_STEPS']),
        'loss_name': str(training_cfg.get('LOSS_NAME', 'bce')),
        'selection_metric': str(training_cfg.get('SELECTION_METRIC', 'Acc@1')),
        'best_epoch': None if training_cfg.get('BEST_EPOCH') is None else int(training_cfg['BEST_EPOCH']),
        'used_eval_split': bool(training_cfg.get('USED_EVAL_SPLIT', False)),
        'num_train_pairs': int(training_cfg.get('NUM_TRAIN_PAIRS', 0)),
        'num_eval_pairs': int(training_cfg.get('NUM_EVAL_PAIRS', 0)),
        'pretrain_topk': int(retrieval_cfg['PRETRAIN_TOPK']),
        'pretrain_num_negatives': int(retrieval_cfg['PRETRAIN_NUM_NEGATIVES']),
        'query_batch_size': int(retrieval_cfg['QUERY_BATCH_SIZE']),
        'dense_vocab_batch_size': int(retrieval_cfg['DENSE_VOCAB_BATCH_SIZE']),
        'st_encode_batch_size': int(retrieval_cfg['ST_ENCODE_BATCH_SIZE']),
        'deduplicate_by_cui': bool(retrieval_cfg['DEDUPLICATE_BY_CUI']),
        'candidate_context_enabled': bool(candidate_context_cfg['ENABLED']),
        'candidate_context_max_aliases': int(candidate_context_cfg['MAX_ALIASES']),
        'candidate_context_num_workers': int(candidate_context_cfg.get('NUM_WORKERS', 1)),
        'candidate_context_alias_length_threshold': candidate_context_cfg['METADATA'].get('threshold') if candidate_context_cfg.get('METADATA') else None,
        'max_pseudo_queries_per_cui': int(pretrain_cfg['MAX_PSEUDO_QUERIES_PER_CUI']),
        'validation_fraction': float(pretrain_cfg['VALIDATION_FRACTION']),
        'pairwise_num_workers': int(pretrain_cfg.get('PAIRWISE_NUM_WORKERS', 1)),
    }


def log_run_to_mlflow(dataset_name, base_cfg, effective_cfg, split_metrics, artifact_paths, training_result):
    experiment_name = f'{MLFLOW_EXPERIMENT_PREFIX}-{dataset_name.lower()}'
    mlflow.set_experiment(experiment_name)
    run_name = MLFLOW_RUN_NAME_TEMPLATE.format(dataset_name=dataset_name.lower())

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(build_mlflow_params(dataset_name, base_cfg, effective_cfg))
        mlflow_metrics = {}
        for split_name, metrics in split_metrics.items():
            for metric_name, metric_value in metrics.items():
                if isinstance(metric_value, (int, float)):
                    normalized_name = metric_name.lower().replace('@', '_at_')
                    mlflow_metrics[f'{split_name}_{normalized_name}'] = float(metric_value)
        if mlflow_metrics:
            mlflow.log_metrics(mlflow_metrics)

        history_df = training_result['history_df'].copy()
        if not history_df.empty:
            if 'step' not in history_df.columns:
                history_df['step'] = range(1, len(history_df) + 1)
            dev_acc_history_columns = [
                column_name
                for column_name in history_df.columns
                if isinstance(column_name, str) and column_name.startswith('dev_Acc@')
            ]
            for _, row in history_df.iterrows():
                step = int(row['step']) if 'step' in row and row['step'] == row['step'] else None
                if 'loss' in row and row['loss'] == row['loss']:
                    mlflow.log_metric('train_loss', float(row['loss']), step=step)
                if 'eval_loss' in row and row['eval_loss'] == row['eval_loss']:
                    mlflow.log_metric('dev_loss', float(row['eval_loss']), step=step)
            if dev_acc_history_columns and 'epoch_int' in history_df.columns:
                dev_acc_history_df = history_df.dropna(subset=['epoch_int']).copy()
                dev_acc_history_df = dev_acc_history_df.dropna(how='all', subset=dev_acc_history_columns)
                if not dev_acc_history_df.empty:
                    dev_acc_history_df['epoch_int'] = dev_acc_history_df['epoch_int'].astype(int)
                    dev_acc_history_df = dev_acc_history_df.sort_values(['epoch_int', 'step'], kind='stable')
                    dev_acc_history_df = dev_acc_history_df.groupby('epoch_int', as_index=False)[dev_acc_history_columns].last()
                    for _, row in dev_acc_history_df.iterrows():
                        epoch_step = int(row['epoch_int'])
                        for column_name in dev_acc_history_columns:
                            metric_value = row[column_name]
                            if metric_value == metric_value:
                                normalized_name = column_name.removeprefix('dev_').lower().replace('@', '_at_')
                                mlflow.log_metric(f'dev_{normalized_name}_history', float(metric_value), step=epoch_step)

        artifact_dir_map = {
            'best_model': 'model',
            'candidate_context_preview': 'prepared_data',
            'candidate_context_metadata': 'prepared_data',
            'pretrain_concepts_preview': 'prepared_data',
            'pretrain_concepts_metadata': 'prepared_data',
            'pretrain_queries_preview': 'prepared_data',
            'pretrain_queries_metadata': 'prepared_data',
            'pretrain_retriever_cache_preview': 'prepared_data',
            'pretrain_retriever_cache_metadata': 'prepared_data',
            'pretrain_pair_examples_preview': 'prepared_data',
            'pretrain_pair_examples_metadata': 'prepared_data',
            'metrics_summary': 'metrics',
            'split_metrics': 'metrics',
            'base_config': 'configs',
            'effective_config': 'configs',
            'training_history': 'training',
            'train_examples': 'training',
        }

        for artifact_name, artifact_path in artifact_paths.items():
            target_dir = artifact_dir_map.get(artifact_name)
            if target_dir is None:
                continue
            if Path(artifact_path).is_dir():
                mlflow.log_artifacts(artifact_path, artifact_path=target_dir)
            else:
                mlflow.log_artifact(artifact_path, artifact_path=target_dir)

        return mlflow.active_run().info.run_id


## Artifacts Utils

In [ ]:
def save_local_artifacts(
    dataset_name,
    base_cfg,
    effective_cfg,
    candidate_context_df,
    candidate_context_metadata,
    concepts_df,
    concepts_metadata,
    queries_df,
    queries_metadata,
    retriever_cache_preview_df,
    retriever_metadata,
    pairwise_df,
    pairwise_metadata,
    split_metrics,
    training_result,
):
    artifact_dir = get_artifact_dir(dataset_name)

    metrics_table_path = artifact_dir / 'metrics_summary.tsv'
    split_metrics_path = artifact_dir / 'split_metrics.json'
    training_history_path = artifact_dir / 'training_history.tsv'
    train_examples_path = artifact_dir / 'train_examples.tsv'
    base_config_path = artifact_dir / 'base_config.json'
    effective_config_path = artifact_dir / 'effective_config.json'
    best_model_path = artifact_dir / 'best_model'

    candidate_context_preview_path = artifact_dir / 'candidate_context_preview.tsv'
    candidate_context_metadata_path = artifact_dir / 'candidate_context_runtime_metadata.json'

    concepts_preview_path = artifact_dir / 'pretrain_concepts_preview.tsv'
    concepts_metadata_path = artifact_dir / 'pretrain_concepts_runtime_metadata.json'

    queries_preview_path = artifact_dir / 'pretrain_queries_preview.tsv'
    queries_metadata_path = artifact_dir / 'pretrain_queries_runtime_metadata.json'

    retriever_preview_path = artifact_dir / 'pretrain_retriever_cache_preview.tsv'
    retriever_metadata_path = artifact_dir / 'pretrain_retriever_cache_runtime_metadata.json'

    pairwise_preview_path = artifact_dir / 'pretrain_pair_examples_preview.tsv'
    pairwise_metadata_path = artifact_dir / 'pretrain_pair_examples_runtime_metadata.json'

    if candidate_context_df is not None:
        serializable_candidate_df = candidate_context_df.copy()
        if 'selected_aliases' in serializable_candidate_df.columns:
            serializable_candidate_df['selected_aliases'] = serializable_candidate_df['selected_aliases'].map(
                lambda values: ' | '.join(values) if isinstance(values, list) else str(values)
            )
        if 'selected_aliases_normalized' in serializable_candidate_df.columns:
            serializable_candidate_df['selected_aliases_normalized'] = serializable_candidate_df['selected_aliases_normalized'].map(
                lambda values: ' | '.join(values) if isinstance(values, list) else str(values)
            )
        if 'languages' in serializable_candidate_df.columns:
            serializable_candidate_df['languages'] = serializable_candidate_df['languages'].map(
                lambda values: ','.join(values) if isinstance(values, list) else str(values)
            )
        serializable_candidate_df.head(200).to_csv(candidate_context_preview_path, sep='\t', index=False)

    serializable_concepts_df = concepts_df.copy()
    if 'pseudo_queries' in serializable_concepts_df.columns:
        serializable_concepts_df['pseudo_queries'] = serializable_concepts_df['pseudo_queries'].map(
            lambda values: ' | '.join(values) if isinstance(values, list) else str(values)
        )
    serializable_concepts_df.head(200).to_csv(concepts_preview_path, sep='\t', index=False)
    queries_df.head(500).to_csv(queries_preview_path, sep='\t', index=False)
    retriever_cache_preview_df.head(1000).to_csv(retriever_preview_path, sep='\t', index=False)
    pairwise_df.head(1000).to_csv(pairwise_preview_path, sep='\t', index=False)

    training_result['history_df'].to_csv(training_history_path, sep='\t', index=False)
    pairwise_df.to_csv(train_examples_path, sep='\t', index=False)

    split_metrics_rows = [
        {'split': split_name, **metrics}
        for split_name, metrics in split_metrics.items()
    ]
    pd.DataFrame(split_metrics_rows).to_csv(metrics_table_path, sep='\t', index=False)

    save_json(split_metrics, split_metrics_path)
    save_json(candidate_context_metadata, candidate_context_metadata_path)
    save_json(concepts_metadata, concepts_metadata_path)
    save_json(queries_metadata, queries_metadata_path)
    save_json(retriever_metadata, retriever_metadata_path)
    save_json(pairwise_metadata, pairwise_metadata_path)
    save_json(base_cfg, base_config_path)
    save_json(effective_cfg, effective_config_path)

    if best_model_path.exists():
        shutil.rmtree(best_model_path)
    shutil.copytree(Path(effective_cfg['TRAINING']['BEST_MODEL_DIR']), best_model_path)

    return {
        'artifact_dir': str(artifact_dir),
        'best_model': str(best_model_path),
        'candidate_context_preview': str(candidate_context_preview_path),
        'candidate_context_metadata': str(candidate_context_metadata_path),
        'pretrain_concepts_preview': str(concepts_preview_path),
        'pretrain_concepts_metadata': str(concepts_metadata_path),
        'pretrain_queries_preview': str(queries_preview_path),
        'pretrain_queries_metadata': str(queries_metadata_path),
        'pretrain_retriever_cache_preview': str(retriever_preview_path),
        'pretrain_retriever_cache_metadata': str(retriever_metadata_path),
        'pretrain_pair_examples_preview': str(pairwise_preview_path),
        'pretrain_pair_examples_metadata': str(pairwise_metadata_path),
        'metrics_summary': str(metrics_table_path),
        'split_metrics': str(split_metrics_path),
        'training_history': str(training_history_path),
        'train_examples': str(train_examples_path),
        'base_config': str(base_config_path),
        'effective_config': str(effective_config_path),
    }


## RU Experiment

In [ ]:
RU_PRETRAIN_CONFIG = build_base_experiment_config(
    reranker_model_name_or_path='andorei/BERGAMOT-multilingual-GAT',
    dataset_name='ru',
)
RU_PRETRAIN_CONFIG['RETRIEVER_MODEL_NAME_OR_PATH'] = "andorei/BERGAMOT-multilingual-GAT"
RU_PRETRAIN_CONFIG


In [ ]:
ru_retrieval_vocab = enrich_vocab_with_oov_train_dev_terms(normalized_vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT)
ru_retrieval_vocab = prepare_experiment_vocab(ru_retrieval_vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_PRETRAIN_CONFIG['RETRIEVAL'])

ru_pretrain_query_vocab, ru_vocab_subset_stats = build_dictionary_pretrain_vocab_subset(
    ru_retrieval_vocab,
    RU_ENTITIES_FOR_VOCAB_ENRICHMENT,
    num_extra_cuis=RU_PRETRAIN_CONFIG['PRETRAIN']['NUM_EXTRA_RANDOM_CUIS'],
    seed=RU_PRETRAIN_CONFIG['TRAINING']['SEED'],
)


In [ ]:
ru_candidate_context_vocab = prepare_candidate_context_vocab(
    enrich_vocab_with_oov_train_dev_terms(
        raw_vocab,
        RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
        text_column='raw_text',
    ),
    RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
    RU_PRETRAIN_CONFIG,
)


In [ ]:
print('RU query vocab:', ru_pretrain_query_vocab.shape)
print('RU retrieval vocab:', ru_retrieval_vocab.shape)
print('RU candidate-context vocab:', ru_candidate_context_vocab.shape)
print('RU vocab subset stats:', ru_vocab_subset_stats)

In [ ]:
ru_pretraining_bundle = run_pretraining(
    dataset_name='ru',
    query_vocab_df=ru_pretrain_query_vocab,
    retrieval_vocab_df=ru_retrieval_vocab,
    candidate_context_vocab_df=ru_candidate_context_vocab,
    cfg=RU_PRETRAIN_CONFIG,
)

ru_runtime_cfg = ru_pretraining_bundle['runtime_cfg']
ru_training_result = ru_pretraining_bundle['training_result']
ru_candidate_context_df = ru_pretraining_bundle['candidate_context_df']
ru_candidate_context_metadata = ru_pretraining_bundle['candidate_context_metadata']
ru_concepts_df = ru_pretraining_bundle['concepts_df']
ru_queries_df = ru_pretraining_bundle['queries_df']
ru_pairwise_df = ru_pretraining_bundle['pairwise_df']
ru_cache_artifacts = ru_pretraining_bundle['cache_artifacts']
ru_cache_metadata = ru_pretraining_bundle['cache_metadata']

ru_retriever_cache_preview_df = pd.read_csv(ru_cache_artifacts['pretrain_retriever_cache']['preview_path'], sep='	')
ru_training_result['best_model_dir']


In [ ]:
RU_EFFECTIVE_CONFIG = build_effective_experiment_config(
    base_cfg=RU_PRETRAIN_CONFIG,
    training_result=ru_training_result,
    candidate_artifact_paths=ru_cache_artifacts['candidate_context'],
    candidate_metadata=ru_candidate_context_metadata,
    pretrain_cache_artifacts=ru_cache_artifacts,
    pretrain_cache_metadata=ru_cache_metadata,
    pretrain_metrics={},
)
RU_EFFECTIVE_CONFIG


In [ ]:
ru_split_metrics = {
    'train': {
        **summarize_pairwise_split(ru_pairwise_df, 'train'),
        'RetrieverHitRate': compute_retriever_hit_rate(ru_pairwise_df, 'train'),
    },
    'dev': {
        **summarize_pairwise_split(ru_pairwise_df, 'dev'),
        'RetrieverHitRate': compute_retriever_hit_rate(ru_pairwise_df, 'dev'),
    },
}
if ru_training_result.get('best_metrics'):
    ru_split_metrics['dev'].update(ru_training_result['best_metrics'])
RU_EFFECTIVE_CONFIG['PRETRAIN']['METRICS'] = ru_split_metrics
ru_split_metrics


In [ ]:
ru_artifact_paths = save_local_artifacts(
    dataset_name='ru',
    base_cfg=RU_PRETRAIN_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    candidate_context_df=ru_candidate_context_df,
    candidate_context_metadata=ru_candidate_context_metadata,
    concepts_df=ru_concepts_df,
    concepts_metadata=ru_cache_metadata['pretrain_concepts'],
    queries_df=ru_queries_df,
    queries_metadata=ru_cache_metadata['pretrain_queries'],
    retriever_cache_preview_df=ru_retriever_cache_preview_df,
    retriever_metadata=ru_cache_metadata['pretrain_retriever_cache'],
    pairwise_df=ru_pairwise_df,
    pairwise_metadata=ru_cache_metadata['pretrain_pair_examples'],
    split_metrics=ru_split_metrics,
    training_result=ru_training_result,
)
ru_artifact_paths


In [ ]:
ru_mlflow_run_id = log_run_to_mlflow(
    dataset_name='ru',
    base_cfg=RU_PRETRAIN_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    split_metrics=ru_split_metrics,
    artifact_paths=ru_artifact_paths,
    training_result=ru_training_result,
)
ru_artifact_paths, ru_mlflow_run_id


In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Fine-Tune Usage

After dictionary pretraining, take the checkpoint path from `ru_artifact_paths['best_model']` or `RU_EFFECTIVE_CONFIG['TRAINING']['BEST_MODEL_DIR']` and use it as `RERANKER_MODEL_NAME_OR_PATH` in `bionnel-cross-encoder-reranking.ipynb`.

The comparison is run in two modes:

- `base cross-encoder -> fine-tune`
- `dictionary-pretrained cross-encoder -> fine-tune`
